In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime
# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # 컬럼 전체 보기
pd.set_option('display.width', 0) 

#데이터 불러오기 
df = pd.read_csv('./model_df.csv')
print(df.columns)
df.head(1)

Index(['기획년도', '주차', '카테고리', '라인', '시즌이월', '시즌', '복종', '소품종', '성별', '총입고수량',
       '판매수량', '판매액', '평균택가', '평균원가', '총입고원가', '총입고택가', '매출원가계', '판매택가계',
       '주차별_평균_실판매가', '월', '월별_평균_실판매가', '시즌별_평균_실판매가', '실판매가', '할인율',
       '누적판매수량', '누적판매액', '누적매출원가', '누적판매택가', '누적판매율', 'ROI', '맑음', '흐림', '비',
       '강한 비', '눈', '강한 눈', '진눈깨비', '악천후일수', '평균기온(도)'],
      dtype='object')


,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도)
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,53,52297400,999000,204500,327200000,1598400000,10838500,52947000,987706.0,1,530682.0,455693,986743.0,1.0,53,52297400,10838500,52947000,3.31,0.11,4,0,0,0,3,0,0,0,-9.1


In [2]:
# 주차 기준 정렬 먼저
df = df.sort_values(['카테고리', '주차'])

# 할인율이 음수인 경우, 같은 카테고리 내에서 ffill
df['할인율'] = df.groupby('카테고리')['할인율'].transform(
    lambda x: x.mask(x < 0).ffill()
)
# df[df['할인율']<0]

In [3]:

# 피벗테이블로 4년동안 데이터가 있는 카테고리만 남김
# 카테고리별 연도 존재 여부 확인 (0: 없음, 1: 있음)
category_year_table = df.groupby(["카테고리", "기획년도"]).size().unstack(fill_value=0)
# 연도가 존재하면 1로 변환 (카테고리가 존재했음을 의미)
category_year_table = (category_year_table > 0).astype(int)

# 21, 22, 23, 24년도 모두 존재한 카테고리만 필터링
categories_all_years = category_year_table[
    (category_year_table.get(2021, 0) == 1) & 
    (category_year_table.get(2022, 0) == 1) & 
    (category_year_table.get(2023, 0) == 1) & 
    (category_year_table.get(2024, 0) == 1)
].index

# 새로운 데이터프레임 생성
df = df[df["카테고리"].isin(categories_all_years)].copy()
# df['카테고리'].nunique()

# 2023년도 데이터(지난 1년동안의 데이터)로 카테고리 선정하고자 함
# 판매 중간에 연도가 바뀌면서 잘린 겨울 제품 제외함함
df = df[df['기획년도'] == 2023]
# df[(df['기획년도'] == 2023) & (df['시즌'] != '겨울')]
# df['카테고리'].nunique()
df

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,판매수량,판매액,평균택가,평균원가,총입고원가,총입고택가,매출원가계,판매택가계,주차별_평균_실판매가,월,월별_평균_실판매가,시즌별_평균_실판매가,실판매가,할인율,누적판매수량,누적판매액,누적매출원가,누적판매택가,누적판매율,ROI,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도)
5516,2023,2023-08-13,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,22,1068000,69900,5436,34301160,441069000,119592,1537800,48545.0,8,48822.0,34667,48545.0,31.0,22,1068000,119592,1537800,0.35,0.02,5,2,0,0,0,0,0,0,28.2
5563,2023,2023-08-20,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,125,6115000,69900,5436,34301160,441069000,679500,8737500,48920.0,8,48822.0,34667,48920.0,30.0,147,7183000,799092,10275300,2.33,0.17,1,3,1,2,0,0,0,2,26.9
5617,2023,2023-08-27,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,55,2695000,69900,5436,34301160,441069000,298980,3844500,49000.0,8,48822.0,34667,49000.0,30.0,202,9878000,1098072,14119800,3.20,0.23,3,1,2,1,0,0,0,1,24.2
5676,2023,2023-09-03,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,134,6556000,69900,5436,34301160,441069000,728424,9366600,48925.0,9,43191.0,34667,48925.0,30.0,336,16434000,1826496,23486400,5.32,0.38,4,3,0,0,0,0,0,0,26.7
5735,2023,2023-09-10,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,104,5086000,69900,5436,34301160,441069000,565344,7269600,48904.0,9,43191.0,34667,48904.0,30.0,440,21520000,2391840,30756000,6.97,0.50,4,1,1,1,0,0,0,1,24.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5611,2023,2023-08-27,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,102827,1867,50321090,99900,9522,979067280,10272417300,17776640,186513300,27044.0,8,26989.0,43143,26953.0,73.0,66887,2475867576,635370680,6682011300,65.05,1.65,3,1,2,1,0,0,0,1,24.2
5657,2023,2023-09-03,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,102827,1399,40163986,99900,9522,979067280,10272417300,13320578,139760100,29204.0,9,27669.0,43143,28709.0,71.0,68286,2516031562,648691258,6821771400,66.41,1.67,4,3,0,0,0,0,0,0,26.7
5717,2023,2023-09-10,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,102827,1393,35938366,99900,9522,979067280,10272417300,13263450,139160700,26773.0,9,27669.0,43143,25799.0,74.0,69679,2551969928,661954708,6960932100,67.76,1.69,4,1,1,1,0,0,0,1,24.1
5780,2023,2023-09-17,여름_팬츠_팬츠(일반)_ZB,ZB,01_시즌,여름,팬츠,팬츠(일반),1:남성,102827,724,19215457,99900,9522,979067280,10272417300,6893566,72327600,27249.0,9,27669.0,43143,26541.0,73.0,70403,2571185385,668848274,7033259700,68.47,1.70,4,1,2,0,0,0,0,0,22.4


In [4]:
# 기준 1) ----------------------------------------------------------------------------------------------------------------
# 매출 기여도가 높은 핵심 카테고리 (판매량 & 매출 기준)
top_sales = df.groupby("카테고리", group_keys=False).agg(
    총판매수량=("판매수량", "sum"),
    총판매액=('판매액', 'sum'),
    총매출원가=('매출원가계', 'sum'),    
    총입고수량=("총입고수량", "max")
).reset_index()
top_sales['총순수익'] = top_sales['총판매액'] - top_sales['총매출원가']

# 기준 2) ----------------------------------------------------------------------------------------------------------------
# 재고 부담 & ROI 개선이 필요한 카테고리 (최대 누적판매율)
top_inventory_ROI = df.groupby("카테고리", group_keys=False).agg(
    최대누적판매율=("누적판매율", "max"),  
    평균할인율=("할인율", "mean"),
    ROI=("ROI", 'max')
).reset_index()

# 기준 3) ---------------------------------------------------------------------------------------------------------
# 할인 전략 개선이 필요한 카테고리(평균 할인율 변화 기준)
# 시즌 종료 시 할인율 상승 패턴 확인 (평균 비교 방식)
df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 기준 3) ---------------------------------------------------------------------------------------------------------
# 할인 전략 개선이 필요한 카테고리 (평균 할인율 변화 기준)
# 시즌 종료 시 할인율 상승 패턴 확인 (평균 비교 방식)

df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 마지막 한달간 할인율 증가량
def calc_discount_diff(group):
    weekly_discount = group.groupby("연도주차", as_index=False)["할인율"].mean()
    last_4 = weekly_discount.tail(4)
    before = weekly_discount.iloc[:-4]
    last_mean = last_4['할인율'].mean()
    before_mean = before['할인율'].mean() if not before.empty else 0
    return pd.Series({
        "마지막4주_평균할인율": last_mean,
        "이전_평균할인율": before_mean,
        "시즌종료_할인율증가량": last_mean - before_mean
    })
discount_rise = df.groupby("카테고리", group_keys=False).apply(calc_discount_diff).reset_index()

# 마지막 주차 할인율
last_week_discount = df.loc[df.groupby("카테고리")["주차"].idxmax()][["카테고리", "할인율"]]
last_week_discount = last_week_discount.rename(columns={"할인율": "마지막주차_할인율"})

# 기준 4) ----------------------------------------------------------------------------------------------------------------
# 가격탄력성이 높은 카테고리 (할인율-판매량 상관관계)
top_correlation = df.groupby("카테고리", group_keys=False).apply(
    lambda x: x["할인율"].corr(x["판매수량"])
).reset_index()
top_correlation.columns = ["카테고리", "가격탄력성(할인율-판매량_상관관계)"]

# 가격_변화율과 판매량_변화율을 이용한 가격 탄력성 계산
df["가격_변화율"] = df.groupby('카테고리')['주차별_평균_실판매가'].pct_change() * 100
df["판매량_변화율"] = df.groupby('카테고리')['판매수량'].pct_change() * 100

df["가격탄력성(판매량변화율/가격변화율)"] = df["판매량_변화율"] / df["가격_변화율"]
df["가격탄력성(판매량변화율/가격변화율)"] = df["가격탄력성(판매량변화율/가격변화율)"].replace([np.inf, -np.inf], np.nan)

elastic_cate = df.groupby('카테고리')['가격탄력성(판매량변화율/가격변화율)'].mean().reset_index()

# 모든 데이터 병합 ----------------------------------------------------------------------------------------------------------------
category_selection = (
    top_sales
    .merge(top_inventory_ROI, on="카테고리", how="left")
    .merge(last_week_discount, on="카테고리", how="left")
    .merge(top_correlation, on="카테고리", how="left")
    .merge(discount_rise, on="카테고리", how="left")
    .merge(elastic_cate, on="카테고리", how="left")
    .fillna(0)  
)

# 소수점 라운딩
category_selection["평균할인율"] = category_selection["평균할인율"].round(2)
category_selection["가격탄력성(할인율-판매량_상관관계)"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].round(2)
category_selection["가격탄력성(판매량변화율/가격변화율)"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].round(2)
category_selection["시즌종료_할인율증가량"] = category_selection["시즌종료_할인율증가량"].astype(int)
category_selection["마지막주차_할인율"] = category_selection["마지막주차_할인율"].astype(int)

# 정렬 및 순위 계산 (최대 누적판매율은 낮을수록 우선순위 → 오름차순, 나머지는 높을수록 우선순위 → 내림차순)
category_selection["총매출_순위"] = category_selection["총판매액"].rank(method="min", ascending=False).astype(int)
category_selection["총순수익_순위"] = category_selection["총순수익"].rank(method="min", ascending=False).astype(int)
category_selection["총입고_순위"] = category_selection["총입고수량"].rank(method="min", ascending=False).astype(int)
category_selection["최대누적판매율_순위"] = category_selection["최대누적판매율"].rank(method="min", ascending=True).astype(int)  # 낮을수록 재고 부담 ↑
category_selection["ROI_순위"] = category_selection["ROI"].rank(method="min", ascending=True).astype(int)  # 낮을수록 수익 손해 ↑
category_selection["평균할인율_순위"] = category_selection["평균할인율"].rank(method="min", ascending=False).astype(int)
category_selection["시즌종료_할인율증가량_순위"] = category_selection["시즌종료_할인율증가량"].rank(method="min", ascending=False).astype(int)
category_selection["가격탄력성(상관계수) 순위"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].rank(method="min", ascending=False).astype(int) 
category_selection["가격탄력성(판매량변화율) 순위"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].rank(method="min", ascending=True).astype(int) # 낮을수록 탄력성 좋음음

# 각 기준별 점수 계산 ----------------------------------------------------------------------------------------------------------------
# 기준 1: 매출 기여도 높은 카테고리 (총매출, 총입고수량)
category_selection["기준1_점수"] = category_selection["총매출_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["총입고_순위"].rank(method="min", ascending=True).astype(int)
                                
# 기준 2: 재고 부담 & ROI 낮은 카테고리 (누적판매율, ROI)
category_selection["기준2_점수"] = category_selection["최대누적판매율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["ROI_순위"].rank(method="min", ascending=True).astype(int)

# 기준 3: 할인 전략 개선이 필요한 카테고리 (최대누적판매율, 평균할인율, 시즌종료_할인율증가량)
category_selection["기준3_점수"] = category_selection["평균할인율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["시즌종료_할인율증가량_순위"].rank(method="min", ascending=True).astype(int)
     
# 기준 4: 가격탄력성이 높아 할인율 최적화 효율이 좋은 카테고리 (상관계수와 판매량변화율로 본 가격탄력성)
category_selection["기준4_점수"] = category_selection["가격탄력성(상관계수) 순위"].rank(method="min", ascending=True).astype(int) + \
                                 category_selection["가격탄력성(판매량변화율) 순위"].rank(method="min", ascending=True).astype(int)

# 기준별 순위 계산 (오름차순, 낮을수록 우선순위)
category_selection["기준1_순위"] = category_selection["기준1_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준2_순위"] = category_selection["기준2_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준3_순위"] = category_selection["기준3_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준4_순위"] = category_selection["기준4_점수"].rank(method="min", ascending=True).astype(int)

# 최종 순위 계산 (기준 1, 2, 3, 4 순위를 모두 합산)
category_selection["최종_순위"] = category_selection["기준1_순위"] + category_selection["기준2_순위"] + category_selection["기준3_순위"] + category_selection["기준4_순위"]
category_selection["최종_순위"] = category_selection["최종_순위"].rank(method="min", ascending=True).astype(int)

# 최종 출력: 순위만 포함 ----------------------------------------------------------------------------------------------------------------
final_rank_output = category_selection[[
    "카테고리","기준1_순위", "기준2_순위", "기준3_순위", "기준4_순위", "최종_순위","총판매수량","총판매액","총순수익","총입고수량","최대누적판매율","ROI","평균할인율","마지막주차_할인율",
    "시즌종료_할인율증가량","가격탄력성(할인율-판매량_상관관계)","가격탄력성(판매량변화율/가격변화율)","총매출_순위","총순수익_순위","총입고_순위","최대누적판매율_순위","ROI_순위",
    "평균할인율_순위","시즌종료_할인율증가량_순위","가격탄력성(상관계수) 순위","가격탄력성(판매량변화율) 순위"]].sort_values(by="최종_순위")

display(final_rank_output) #최종순위대로 정렬
# 기준 점수를 기준으로 정렬, 하나씩 주석 해제하면서 정렬
# final_rank_output.sort_values(by=["기준1_순위"])
# final_rank_output.sort_values(by=["기준2_순위"])
# final_rank_output.sort_values(by=["기준3_순위"])
# final_rank_output.sort_values(by=["기준4_순위"])

,카테고리,기준1_순위,기준2_순위,기준3_순위,기준4_순위,최종_순위,총판매수량,총판매액,총순수익,총입고수량,최대누적판매율,ROI,평균할인율,마지막주차_할인율,시즌종료_할인율증가량,가격탄력성(할인율-판매량_상관관계),가격탄력성(판매량변화율/가격변화율),총매출_순위,총순수익_순위,총입고_순위,최대누적판매율_순위,ROI_순위,평균할인율_순위,시즌종료_할인율증가량_순위,가격탄력성(상관계수) 순위,가격탄력성(판매량변화율) 순위
5,겨울_사파리_패딩사파리_ZB,6,3,5,8,1,18883,2584464090,1512256101,44038,42.88,0.51,49.16,74,28,0.90,-13.69,7,7,8,10,1,15,2,4,20
21,겨울_코트_싱글코트_ZB,10,3,13,6,2,12027,2074470226,1271735376,28165,42.70,0.58,40.19,58,22,0.90,-18.12,8,12,15,9,2,25,6,4,17
18,겨울_점퍼_패딩점퍼_ZB,19,1,13,9,3,9013,882041628,640764498,24638,36.58,0.85,48.67,67,17,0.85,-17.90,25,25,17,1,5,16,15,9,18
23,겨울_팬츠_팬츠(일반)_ZB,11,15,31,5,4,24574,1355779092,1007315437,52873,46.48,1.18,38.79,49,12,0.74,-421.60,18,19,6,14,17,32,28,15,2
45,여름_우븐 셔츠_캐쥬얼셔츠_ZB,12,34,5,12,5,37990,1078654234,821401310,60865,62.42,1.75,64.58,79,16,0.49,-116.28,22,22,4,32,41,1,16,28,4
15,겨울_스웨터_라운드_ZB,17,3,17,29,6,14774,781959566,567712164,38348,38.53,0.89,43.05,61,18,0.70,8.59,29,28,10,4,7,22,13,17,38
50,여름_팬츠_팬츠(일반)_ZB,2,42,8,15,7,70962,2586460323,1912289531,102827,69.01,1.71,55.79,73,19,0.63,-21.62,6,5,2,41,40,10,11,22,16
13,겨울_스웨터_Turtle_ZB,31,9,24,3,7,8504,439960397,318990873,20770,40.94,0.94,40.15,56,16,0.92,-28.99,39,37,20,6,11,26,16,2,14
19,겨울_코트_더블코트_ZA,42,1,26,1,9,1746,454292629,302913925,4537,38.48,0.67,41.20,53,14,0.86,-172.92,37,38,48,3,3,24,23,8,3
46,여름_자켓_싱글재킷_ZA,7,44,12,14,10,20913,3129651456,2322980090,30109,69.46,1.75,47.76,67,19,0.20,-1038.68,2,2,14,42,41,18,11,35,1


In [5]:
final_rank_output.to_csv("category_rank(소품종,겨울포함).csv", index=False, encoding="utf-8-sig")